In [1]:
import os
os.chdir(os.path.join(os.path.abspath(""), ".."))
os.getcwd()

'/home/harim/Desktop/kaist/DS801/StockMixer'

In [2]:
from models.StockMixer import StockMixer, get_loss
from einops import rearrange
from utils import set_seed, evaluate
import torch
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader
from data import StockDataset

In [3]:
train_dataset = StockDataset(seq_len=16, mode="train")
val_dataset = StockDataset(seq_len=16, mode="valid")
test_dataset = StockDataset(seq_len=16, mode="test")

train_dl = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_dl = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_dl = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [4]:
seed = 1
set_seed(seed)
device = torch.device("cuda") if torch.cuda.is_available() else 'cpu'

stock_num = train_dataset.stock_dim
feature_dim = train_dataset.feature_dim
seq_len = train_dataset.seq_len

In [5]:
alpha = 0.1
scale_factor = 8
market_num = 20

model = StockMixer(
    stocks=stock_num,
    time_steps=seq_len,
    channels=feature_dim,
    market=market_num,
    scale_dim=scale_factor
    ).to(device)

learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [6]:
epochs = 100
best_valid_loss = np.inf
best_test_RIC = -np.inf
best_valid_perf = None
best_test_perf = None


In [7]:
for epoch in range(epochs):
    model.train()
    pbar = tqdm(train_dl, desc="Train")
    for data_batch, mask_batch, base_batch, target_batch in pbar:
        data_batch, mask_batch, base_batch, target_batch = data_batch.squeeze(0).to(device), mask_batch.squeeze(0).to(device), base_batch.squeeze(0).to(device), target_batch.squeeze(0).to(device)
        optimizer.zero_grad()
        prediction = model(data_batch)
        loss, reg_loss, rank_loss, _ = get_loss(prediction, target_batch, base_batch, mask_batch, alpha)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=loss.item(), reg_loss=reg_loss.item(), rank_loss=rank_loss.item())
        
    model.eval()
    # validate
    val_pred = []
    for i in tqdm(range(len(val_dataset)), desc="validate"):
        data_batch = val_dataset.eod_data[i].to(device)
        base_batch = val_dataset.base_price[i].to(device)
        prediction = model(data_batch)
        target = (prediction - base_batch) / base_batch
        val_pred.append(target.cpu().detach().numpy())
        
    val_pred = rearrange(val_pred, "T N 1 -> N T")
    target = rearrange(val_dataset.target, "T N 1 -> N T").numpy()
    mask = rearrange(val_dataset.mask_data, "T N 1 -> N T").numpy()
    val_performance = evaluate(val_pred, target, mask)

    # test
    test_pred = []
    for i in tqdm(range(len(test_dataset)), desc="test"):
        data_batch = test_dataset.eod_data[i].to(device)
        base_batch = test_dataset.base_price[i].to(device)
        prediction = model(data_batch)
        target = (prediction - base_batch) / base_batch
        test_pred.append(target.cpu().detach().numpy())
    test_pred = rearrange(test_pred, "T N 1 -> N T")
    target = rearrange(test_dataset.target, "T N 1 -> N T").numpy()
    mask = rearrange(test_dataset.mask_data, "T N 1 -> N T").numpy()
    test_performance = evaluate(test_pred, target, mask)
    
    
    if best_test_RIC < test_performance["RIC"]:
        best_test_RIC = test_performance["RIC"]
        best_test_perf = test_performance
        torch.save(model.state_dict(), "stockmixer.pt")
        
    print(f"Epoch : {epoch}, val_mse : {val_performance['mse']:0.4f}, val_IC : {val_performance['IC']:0.4f}, val_RIC : {val_performance['RIC']:0.4f}, val_sharpe5 : {val_performance['sharpe5']:0.4f}, val_prec_10 : {val_performance['prec_10']:0.4f}")
    print(f"Epoch : {epoch}, test_mse : {best_test_perf['mse']:0.4f}, test_IC : {best_test_perf['IC']:0.4f}, test_RIC : {best_test_perf['RIC']:0.4f}, test_sharpe5 : {best_test_perf['sharpe5']:0.4f}, test_prec_10 : {best_test_perf['prec_10']:0.4f}")
    


test: 100%|██████████| 273/273 [00:00<00:00, 520.74it/s]


Epoch : 0, val_mse : 0.0044, val_IC : 0.0216, val_RIC : 0.2200, val_sharpe5 : 1.4730, val_prec_10 : 0.5341
Epoch : 0, test_mse : 0.0052, test_IC : 0.0227, test_RIC : 0.2667, test_sharpe5 : 1.9557, test_prec_10 : 0.5337


test: 100%|██████████| 273/273 [00:00<00:00, 505.54it/s]


Epoch : 1, val_mse : 0.0037, val_IC : 0.0276, val_RIC : 0.2615, val_sharpe5 : 1.5574, val_prec_10 : 0.5357
Epoch : 1, test_mse : 0.0052, test_IC : 0.0227, test_RIC : 0.2667, test_sharpe5 : 1.9557, test_prec_10 : 0.5337


test: 100%|██████████| 273/273 [00:01<00:00, 250.05it/s]


...

Epoch : 97, val_mse : 0.0009, val_IC : 0.0047, val_RIC : 0.0551, val_sharpe5 : 1.2239, val_prec_10 : 0.5226
Epoch : 97, test_mse : 0.0004, test_IC : 0.0192, test_RIC : 0.2716, test_sharpe5 : 2.9471, test_prec_10 : 0.5443


test: 100%|██████████| 273/273 [00:00<00:00, 774.28it/s]


Epoch : 98, val_mse : 0.0007, val_IC : 0.0351, val_RIC : 0.2895, val_sharpe5 : 2.3603, val_prec_10 : 0.5246
Epoch : 98, test_mse : 0.0004, test_IC : 0.0192, test_RIC : 0.2716, test_sharpe5 : 2.9471, test_prec_10 : 0.5443


test: 100%|██████████| 273/273 [00:00<00:00, 778.58it/s]


Epoch : 99, val_mse : 0.0005, val_IC : -0.0164, val_RIC : -0.1487, val_sharpe5 : 1.0353, val_prec_10 : 0.5115
Epoch : 99, test_mse : 0.0004, test_IC : 0.0192, test_RIC : 0.2716, test_sharpe5 : 2.9471, test_prec_10 : 0.5443
